In [1]:
!pip install ccxt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.8/151.8 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.4/6.4 MB 88.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 21.0 MB/s eta 0:00:00


In [2]:
import csv

def calc_maker_taker_pnl(bid_price, ask_price, size, maker_fee=0.0, taker_fee=0.0004):
    """
    计算一笔双边做市交易的净利润

    参数：
    - bid_price: 买单成交价
    - ask_price: 卖单成交价
    - size: 交易量（BTC）
    - maker_fee: Maker费率（负数=返佣，如-0.0001）
    - taker_fee: Taker费率（正数=收费，如0.0004）
    """
    # 买入：你是Maker，交易所返你钱或免费
    buy_cost = bid_price * size
    maker_rebate = buy_cost * maker_fee  # 负值=返钱，正值=收费
    buy_net = buy_cost + maker_rebate  # +负数 = 实际花更少

    # 卖出：你是Maker，同样返你钱
    sell_proceed = ask_price * size
    maker_rebate_sell = sell_proceed * maker_fee
    sell_net = sell_proceed + maker_rebate_sell  # 卖出拿到的钱 + 返佣

    # 毛利润
    spread = (ask_price - bid_price) * size
    total_rebate = maker_rebate + maker_rebate_sell
    net_pnl = spread + total_rebate

    return {
        'spread_pnl': round(spread, 2),
        'maker_rebate': round(total_rebate, 2),
        'net_pnl': round(net_pnl, 2),
        'buy_net_cost': round(buy_net, 2),
        'sell_net_proceed': round(sell_net, 2),
    }

# 算一笔
result = calc_maker_taker_pnl(
    bid_price=63000,
    ask_price=63050,
    size=1.0,
    maker_fee=-0.0001,   # -0.01%返佣
    taker_fee=0.0004     # 0.04% 收费
)

print(f"价差收益: ${result['spread_pnl']}")
print(f"Maker返佣: ${result['maker_rebate']}")
print(f"净收益:   ${result['net_pnl']}")
# 把多笔交易记到CSV里
import csv
from datetime import datetime

def save_trade_to_csv(trade_data, filename='maker_trades.csv'):
    """保存一笔做市交易到CSV"""
    file_exists = False
    try:
        with open(filename, 'r') as f:
            file_exists = True
    except FileNotFoundError:
        pass

    with open(filename, 'a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['time', 'pair', 'side', 'price', 'size', 'fee', 'net'])
        writer.writerow([
            datetime.now().isoformat(),
            trade_data['pair'],
            trade_data['side'],
            trade_data['price'],
            trade_data['size'],
            trade_data['fee'],
            trade_data['net']
        ])

# 示例：存一笔成交
save_trade_to_csv({
    'pair': 'BTC/USD',
    'side': 'buy',
    'price': 63000,
    'size': 1.0,
    'fee': -0.63,   # 返佣$0.63
    'net': 62999.37
})

价差收益: $50.0
Maker返佣: $-12.61
净收益:   $37.39
